# LLM-powered Airline Customer Feedback Retrieval and Question Answering using RAG

## Project Objective
Develop an intelligent customer feedback retrieval system that enables natural language querying over airline customer feedback using Large Language Models (LLMs), semantic search, and Retrieval-Augmented Generation (RAG). The system retrieves contextually relevant customer feedback from a vector database and generates grounded, business-focused insights to help identify customer concerns, service issues, and opportunities for operational improvement.

## Dataset Description

| **Column Name**                | **Description**                                           |
| ------------------------------ | --------------------------------------------------------- |
| `tweet_id`                     | Unique identifier for each tweet                          |
| `airline_sentiment`            | Sentiment of the tweet: positive, negative, or neutral    |
| `airline_sentiment_confidence` | Confidence score of the sentiment classification (0 to 1) |
| `negativereason`               | Reason for negative sentiment (if applicable)             |
| `negativereason_confidence`    | Confidence of the negative reason label                   |
| `airline`                      | Name of the airline mentioned in the tweet                |
| `name`                         | Twitter username of the person who posted the tweet       |
| `retweet_count`                | Number of times the tweet has been retweeted              |
| `text`                         | Content of the tweet                                      |
| `tweet_coord`                  | Geolocation coordinates of the tweet (if available)       |
| `tweet_created`                | Timestamp when the tweet was posted                       |
| `tweet_location`               | User-reported location (if available)                     |
| `user_timezone`                | Timezone of the user posting the tweet                    |


In [2]:
!pip install sentence-transformers
!pip install faiss-cpu
!pip install langchain
!pip install langchain-community
!pip install langchain-openai
!pip install openai
!pip install transformers
!pip install torch
!pip install tiktoken
!pip install streamlit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 81.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 45.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 52.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 9.4 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 558.3/558.3 kB 30.2 MB/s eta 0:00:00
   ━━━━━━━━━━━

## Import libraries

In [4]:
import pandas as pd
import numpy as np

# Embeddings
from sentence_transformers import SentenceTransformer

# Vector Database
import faiss

# LLM
from openai import OpenAI

# LangChain
from langchain_core.prompts import PromptTemplate

## Load data

In [7]:
data = pd.read_csv(r'/content/Tweets.csv')
data.head(3).T

,0,1,2
tweet_id,570306133677760513,570301130888122368,570301083672813571
airline_sentiment,neutral,positive,neutral
airline_sentiment_confidence,1.0,0.3486,0.6837
negativereason,NaN,NaN,NaN
negativereason_confidence,NaN,0.0,NaN
airline,Virgin America,Virgin America,Virgin America
airline_sentiment_gold,NaN,NaN,NaN
name,cairdin,jnardino,yvonnalynn
negativereason_gold,NaN,NaN,NaN
retweet_count,0,0,0


In [8]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14640 entries, 0 to 14639
Data columns (total 15 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   tweet_id                      14640 non-null  int64  
 1   airline_sentiment             14640 non-null  object 
 2   airline_sentiment_confidence  14640 non-null  float64
 3   negativereason                9178 non-null   object 
 4   negativereason_confidence     10522 non-null  float64
 5   airline                       14640 non-null  object 
 6   airline_sentiment_gold        40 non-null     object 
 7   name                          14640 non-null  object 
 8   negativereason_gold           32 non-null     object 
 9   retweet_count                 14640 non-null  int64  
 10  text                          14640 non-null  object 
 11  tweet_coord                   1019 non-null   object 
 12  tweet_created                 14640 non-null  object 
 13  t

In [ ]:
# missing data
data.isnull().sum()

tweet_id                            0
airline_sentiment                   0
airline_sentiment_confidence        0
negativereason                   5462
negativereason_confidence        4118
airline                             0
airline_sentiment_gold          14600
name                                0
negativereason_gold             14608
retweet_count                       0
text                                0
tweet_coord                     13621
tweet_created                       0
tweet_location                   4733
user_timezone                    4820
dtype: int64

In [9]:
data.isnull().sum()/len(data)*100

,0
tweet_id,0.000000
airline_sentiment,0.000000
airline_sentiment_confidence,0.000000
negativereason,37.308743
negativereason_confidence,28.128415
airline,0.000000
airline_sentiment_gold,99.726776
name,0.000000
negativereason_gold,99.781421
retweet_count,0.000000


All tweets contain text, so missing values in other columns are not treated for this analysis.

## Data Preprocessing

### Preprocessing Functions

In [13]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import re, string, unicodedata
# import contractions
from bs4 import BeautifulSoup

import nltk                   ## Import Natural Language Tool-kit
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('wordnet')


from nltk.corpus import stopwords                       # Import stopwords.
from nltk.tokenize import word_tokenize, sent_tokenize  # Import Tokenizer.
from nltk.stem.wordnet import WordNetLemmatizer         # Import Lemmatizer.

import warnings
warnings.filterwarnings('ignore')


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...


In [21]:
lemmatizer = WordNetLemmatizer()

def clean_text(text):
    if not isinstance(text, str):
        return ""

    # 1. Remove HTML tags
    text = BeautifulSoup(text, "html.parser").get_text()

    # 2. Expand contractions
    text = contractions.fix(text)

    # 3. Remove special characters, numbers, and punctuations
    text = re.sub(r'\d+', '', text)
    text = text.translate(str.maketrans('', '', string.punctuation))

    # 4. Normalize unicode characters
    text = unicodedata.normalize('NFKD', text).encode('ascii', 'ignore').decode('utf-8', 'ignore')

    # 5. Tokenize
    tokens = word_tokenize(text.lower())

    # 6. Remove stopwords
    tokens = [word for word in tokens if word not in stop_words]

    # 7. Lemmatize
    tokens = [lemmatizer.lemmatize(word) for word in tokens]

    # 8. Join tokens back to string
    clean_text = ' '.join(tokens)

    return clean_text


In [16]:
!pip install contractions

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 345.1/345.1 kB 14.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 7.1 MB/s eta 0:00:00


In [17]:
import contractions

In [19]:
import nltk

nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...


True

In [23]:
from nltk.corpus import stopwords

stop_words = set(stopwords.words('english'))

In [24]:
# Apply Preprocessing to Your Dataset
data['text'] = data['text'].apply(clean_text)
data['text'].head()

# data['text'] = data.apply(lambda row: clean_text(row['text']), axis=1)
# data.head()

,text
0,virginamerica dhepburn said
1,virginamerica plus added commercial experience...
2,virginamerica today must mean need take anothe...
3,virginamerica really aggressive blast obnoxiou...
4,virginamerica really big bad thing


In [26]:
text = data.to_csv('cleaned_data.csv', index=False)

In [27]:
df = pd.read_csv(r'/content/cleaned_data.csv')
df

,tweet_id,airline_sentiment,airline_sentiment_confidence,negativereason,negativereason_confidence,airline,airline_sentiment_gold,name,negativereason_gold,retweet_count,text,tweet_coord,tweet_created,tweet_location,user_timezone
0,570306133677760513,neutral,1.0000,NaN,NaN,Virgin America,NaN,cairdin,NaN,0,virginamerica dhepburn said,NaN,2015-02-24 11:35:52 -0800,NaN,Eastern Time (US & Canada)
1,570301130888122368,positive,0.3486,NaN,0.0000,Virgin America,NaN,jnardino,NaN,0,virginamerica plus added commercial experience...,NaN,2015-02-24 11:15:59 -0800,NaN,Pacific Time (US & Canada)
2,570301083672813571,neutral,0.6837,NaN,NaN,Virgin America,NaN,yvonnalynn,NaN,0,virginamerica today must mean need take anothe...,NaN,2015-02-24 11:15:48 -0800,Lets Play,Central Time (US & Canada)
3,570301031407624196,negative,1.0000,Bad Flight,0.7033,Virgin America,NaN,jnardino,NaN,0,virginamerica really aggressive blast obnoxiou...,NaN,2015-02-24 11:15:36 -0800,NaN,Pacific Time (US & Canada)
4,570300817074462722,negative,1.0000,Can't Tell,1.0000,Virgin America,NaN,jnardino,NaN,0,virginamerica really big bad thing,NaN,2015-02-24 11:14:45 -0800,NaN,Pacific Time (US & Canada)
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14635,569587686496825344,positive,0.3487,NaN,0.0000,American,NaN,KristenReenders,NaN,0,americanair thank got different flight chicago,NaN,2015-02-22 12:01:01 -0800,NaN,NaN
14636,569587371693355008,negative,1.0000,Customer Service Issue,1.0000,American,NaN,itsropes,NaN,0,americanair leaving minute late flight warning...,NaN,2015-02-22 11:59:46 -0800,Texas,NaN
14637,569587242672398336,neutral,1.0000,NaN,NaN,American,NaN,sanyabun,NaN,0,americanair please bring american airline blac...,NaN,2015-02-22 11:59:15 -0800,"Nigeria,lagos",NaN
14638,569587188687634433,negative,1.0000,Customer Service Issue,0.6659,American,NaN,SraJackson,NaN,0,americanair money change flight answer phone s...,NaN,2015-02-22 11:59:02 -0800,New Jersey,Eastern Time (US & Canada)


## Load Embedding Model

Sentence Transformers converts text into embeddings.

In [28]:
embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

### Generate Embeddings

In [29]:
df

,tweet_id,airline_sentiment,airline_sentiment_confidence,negativereason,negativereason_confidence,airline,airline_sentiment_gold,name,negativereason_gold,retweet_count,text,tweet_coord,tweet_created,tweet_location,user_timezone
0,570306133677760513,neutral,1.0000,NaN,NaN,Virgin America,NaN,cairdin,NaN,0,virginamerica dhepburn said,NaN,2015-02-24 11:35:52 -0800,NaN,Eastern Time (US & Canada)
1,570301130888122368,positive,0.3486,NaN,0.0000,Virgin America,NaN,jnardino,NaN,0,virginamerica plus added commercial experience...,NaN,2015-02-24 11:15:59 -0800,NaN,Pacific Time (US & Canada)
2,570301083672813571,neutral,0.6837,NaN,NaN,Virgin America,NaN,yvonnalynn,NaN,0,virginamerica today must mean need take anothe...,NaN,2015-02-24 11:15:48 -0800,Lets Play,Central Time (US & Canada)
3,570301031407624196,negative,1.0000,Bad Flight,0.7033,Virgin America,NaN,jnardino,NaN,0,virginamerica really aggressive blast obnoxiou...,NaN,2015-02-24 11:15:36 -0800,NaN,Pacific Time (US & Canada)
4,570300817074462722,negative,1.0000,Can't Tell,1.0000,Virgin America,NaN,jnardino,NaN,0,virginamerica really big bad thing,NaN,2015-02-24 11:14:45 -0800,NaN,Pacific Time (US & Canada)
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14635,569587686496825344,positive,0.3487,NaN,0.0000,American,NaN,KristenReenders,NaN,0,americanair thank got different flight chicago,NaN,2015-02-22 12:01:01 -0800,NaN,NaN
14636,569587371693355008,negative,1.0000,Customer Service Issue,1.0000,American,NaN,itsropes,NaN,0,americanair leaving minute late flight warning...,NaN,2015-02-22 11:59:46 -0800,Texas,NaN
14637,569587242672398336,neutral,1.0000,NaN,NaN,American,NaN,sanyabun,NaN,0,americanair please bring american airline blac...,NaN,2015-02-22 11:59:15 -0800,"Nigeria,lagos",NaN
14638,569587188687634433,negative,1.0000,Customer Service Issue,0.6659,American,NaN,SraJackson,NaN,0,americanair money change flight answer phone s...,NaN,2015-02-22 11:59:02 -0800,New Jersey,Eastern Time (US & Canada)


In [31]:
tweets = df["text"].tolist()

In [32]:
embeddings = embedding_model.encode(
    tweets,
    convert_to_numpy=True
)

In [33]:
print(embeddings.shape)

(14640, 384)


Create Vector Database

In [34]:
dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)

In [35]:
index.add(
    embeddings.astype("float32")
)

In [36]:
print(index.ntotal)

14640


### Save Vector Database

In [37]:
faiss.write_index(
    index,
    "airline_vector_db.faiss"
)

In [38]:
index = faiss.read_index(
    "airline_vector_db.faiss"
)

### Semantic Search

In [40]:
question = "Passengers complaining about baggage"

In [43]:
# Generate embedding
query_vector = embedding_model.encode(
    [question],
    convert_to_numpy=True
)

In [42]:
# Search
distance,indices = index.search(

    query_vector.astype("float32"),

    5
)

In [45]:
# Retrieve tweets
retrieved=[]

for i in indices[0]:

    retrieved.append(

        df.iloc[i]["text"]

    )

In [46]:
# Display
for tweet in retrieved:

    print(tweet)

americanair extremely upset baggage handler decide go luggage take belonging
americanair extremely upset baggage handler decide go luggage take belonging
southwestair along passenger repeatedly asked southwest personnel taking long luggage
united announcement extra baggage find empty bin aisle back baggage seat row agent argueing
virginamerica normal receive reply central baggage baggageissues smh


Prepare RAG Context

In [47]:
context = "\n".join(retrieved)

### Prompt Engineering

In [48]:
template = """
You are an Airline Customer Experience Analyst.

Answer ONLY using the retrieved customer feedback.

Customer Feedback:

{context}

Question:

{question}

Provide a concise business summary.
"""

In [49]:
#Create prompt
prompt = PromptTemplate(

    template=template,

    input_variables=["context","question"]

)

In [51]:
# Format
final_prompt = prompt.format(

    context=context,

    question=question

)

## LLM

In [59]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-base")
model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-base")

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


In [60]:
inputs = tokenizer(
    final_prompt,
    return_tensors="pt",
    truncation=True,
    max_length=512
)


In [61]:
outputs = model.generate(
    **inputs,
    max_new_tokens=150
)

In [62]:
answer = tokenizer.decode(
    outputs[0],
    skip_special_tokens=True
)

print(answer)

Passengers complaining about baggage


## Conclusion

This project successfully implemented an end-to-end Retrieval-Augmented Generation (RAG) pipeline for intelligent airline customer feedback retrieval and analysis. Customer feedback was transformed into sentence embeddings using a transformer-based embedding model and stored in a FAISS vector database to enable efficient semantic search. Relevant feedback was retrieved based on user queries and supplied as contextual information to the FLAN-T5 open-source Large Language Model, which generated concise, context-aware business summaries.

Unlike traditional keyword-based search, the semantic search approach retrieved feedback based on contextual similarity, improving the relevance of retrieved information. The integration of embeddings, vector search, prompt engineering, and an LLM demonstrated how Retrieval-Augmented Generation can provide grounded responses from enterprise data.

This project provides practical exposure to modern Generative AI concepts, including sentence embeddings, vector databases, semantic search, prompt engineering, Large Language Models, and Retrieval-Augmented Generation (RAG), illustrating their application in building intelligent customer feedback retrieval systems.
